# Hamiltonian simulation runner

In [ ]:
# (C) Copyright IBM 2026.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

In [ ]:
import time
import datetime
import json
import numpy as np

from qiskit import *
from supermarq.benchmarks.hamiltonian_simulation import HamiltonianSimulation

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler, Batch

In [ ]:
service = QiskitRuntimeService()

In [ ]:
backend = service.backend("ibm_chicago", use_fractional_gates=True)

In [ ]:
from qiskit_aer import AerSimulator

In [ ]:
sim = AerSimulator(method='matrix_product_state')

In [ ]:
shots = 2**16
shots

In [ ]:
num_qubits = 100
dt = 0.035
steps = 101
circuits = []
gate_counts = []
trans_gate_counts = []
sim_mags = []

for kk in range(steps):
    ham = HamiltonianSimulation(num_qubits, dt, dt*kk)
    qc = ham.qiskit_circuit()
    gate_counts.append(qc.count_ops())
    circuits.append(qc)
    sim_counts = sim.run(qc, shots=shots).result().get_counts()
    sim_mag = ham._average_magnetization(sim_counts, shots=shots)
    sim_mags.append(sim_mag)

sim_mags = np.array(sim_mags)

In [ ]:
sim_mags

In [ ]:
trans_circuits = transpile(circuits, backend, optimization_level=3)

In [ ]:
trans_gate_counts = []
for circ in trans_circuits:
    trans_gate_counts.append(circ.count_ops())

In [ ]:
sampler = Sampler(backend)

In [ ]:
job = sampler.run(trans_circuits, shots=shots)

In [ ]:
counts = [job.result()[kk].join_data().get_counts() for kk in range(steps)]

In [ ]:
hw_mags = np.array([ham._average_magnetization(counts[kk], shots=shots) for kk in range(steps)])

In [ ]:
scores = 1 - np.abs(sim_mags - hw_mags) / 2

In [ ]:
sim_magentizations = [float(item) for item in sim_mags] 
hw_magentizations = [float(item) for item in hw_mags] 
scores = [float(item) for item in scores]
orig_gate_counts = [dict(item) for item in gate_counts]
trans_gate_counts = [dict(item) for item in trans_gate_counts]

In [ ]:
results = {'name': 'hamsim', 'backend': backend.name, 'datetime': str(job.creation_date),
           'qubits': num_qubits, 'dt':dt, 'steps':steps, 'shots': shots, 'runtime': job.usage(),
           'orig_gate_counts': orig_gate_counts, 'trans_gate_counts': trans_gate_counts,
           'sim_magnetizations': sim_magentizations, 'hw_magnetizations':hw_magentizations, 'scores':scores}

In [ ]:
with open(f'hamsim__results_{num_qubits}Q.json', 'w') as fp:
    json.dump(results, fp)

In [ ]:
shots = 2**16

In [ ]:
num_qubits = 16
steps = 100
dt = 0.035

ham = HamiltonianSimulation(num_qubits, dt, dt*steps)
qc = ham.qiskit_circuit()

In [ ]:
st = time.perf_counter()
sim_counts = sim.run(qc, shots=shots).result().get_counts()
sim_mag = ham._average_magnetization(sim_counts, shots=shots)
ft = time.perf_counter()
classical_time = ft -st
classical_time

In [ ]:
trans_circuits = transpile([qc]*10, backend, optimization_level=3)

In [ ]:
sampler = Sampler(backend)

In [ ]:
job = sampler.run(trans_circuits, shots=shots)

In [ ]:
counts = [res.join_data().get_counts() for res in job.result()]

In [ ]:
hw_mags = np.array([ham._average_magnetization(cc, shots=shots) for cc in counts])

In [ ]:
scores = 1 - np.abs(sim_mag - hw_mags) / 2

In [ ]:
np.mean(sim_mag - hw_mags)

In [ ]:
np.mean(scores)

In [ ]:
job.usage()/10